#Dataset Builder 

##Objective

Automatically process all gravitational wave recordings stored in 'data/raw/events' directory.

The pipeline will:
- Load every HDF5 file
- Extract the strain signal
- Apply preprocessing
- Split the signal into windows
- Combine all windows into single machine learning dataset

In [2]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt


In [3]:
def remove_dc(signal):
    return signal - np.mean(signal)

def bandpass_filter(signal, lowcut=20, highcut=400, fs=4096, order=4):
    nyquist = fs/2

    low = lowcut/nyquist
    high = highcut/nyquist

    b, a = butter(order, [low, high], btype="band")

    return filtfilt(b,a,signal)

def normalize(signal):
              return (signal-np.mean(signal))/np.std(signal)

def preprocess(signal):
    signal=remove_dc(signal)
    signal=bandpass_filter(signal)
    signal=normalize(signal)

    return signal

In [4]:
event_folder = "../data/raw/events"
event_files=sorted(
    [
        os.path.join(event_folder, file)
        for file in os.listdir(event_folder)
        if file.endswith(".hdf5")
    ]
)
print("Number of files:", len(event_files))
for file in event_files:
    print(os.path.basename(file))

Number of files: 5
H-H1_GWOSC_O2_4KHZ_R1-1167556608-4096.hdf5
H-H1_GWOSC_O2_4KHZ_R1-1186738176-4096.hdf5
H-H1_LOSC_4_V1-1126256640-4096.hdf5
H-H1_LOSC_4_V1-1135132672-4096.hdf5
L-L1_GWOSC_O2_4KHZ_R1-1180921856-4096.hdf5


In [5]:
def load_strain(filepath):
    """
    Load strain data from a LIGO HDF5 file.
    """
    with h5py.File(filepath, "r") as f:
        strain = f["strain"]["Strain"][:]

    return strain

In [6]:
test_signal=load_strain(event_files[0])
print("Shape:", test_signal.shape)
print("Datatype:", test_signal.dtype)

Shape: (16777216,)
Datatype: float64


In [16]:
def create_windows(signal, window_size=4096):
    windows = []

    for start in range(0, len(signal), window_size):
        end = start + window_size

        if end <= len(signal):
            windows.append(signal[start:end])

    return np.array(windows)

In [17]:
all_windows=[]
for filepath in event_files:
    print(f"\nProcessing: {os.path.basename(filepath)}")

    strain = load_strain(filepath)

    processed = preprocess(strain)

    windows = create_windows(processed)

    print("Number of windows:", windows.shape[0])

    all_windows.append(windows)


Processing: H-H1_GWOSC_O2_4KHZ_R1-1167556608-4096.hdf5
Number of windows: 4096

Processing: H-H1_GWOSC_O2_4KHZ_R1-1186738176-4096.hdf5
Number of windows: 4096

Processing: H-H1_LOSC_4_V1-1126256640-4096.hdf5
Number of windows: 4096

Processing: H-H1_LOSC_4_V1-1135132672-4096.hdf5
Number of windows: 4096

Processing: L-L1_GWOSC_O2_4KHZ_R1-1180921856-4096.hdf5
Number of windows: 4096


In [18]:
dataset = np.vstack(all_windows)
print("Final Datset Shape:", dataset.shape)

Final Datset Shape: (20480, 4096)


In [20]:
print("Dataset shape:",dataset.shape)
print("Data Type:", dataset.dtype)

print("\nFirst window:")
print(dataset[0][:10])

Dataset shape: (20480, 4096)
Data Type: float64

First window:
[6.33030986 7.12983132 7.67826371 7.74588263 7.25408119 6.26045352
 4.9126315  3.39334722 1.8669872  0.44460309]


In [21]:
import os

os.makedirs("../data/datasets", exist_ok=True)

np.save("../data/datasets/event_windows.npy", dataset)

print("Dataset saved successfully!")

Dataset saved successfully!


In [22]:
EVENT_GPS = {
    1126259462: "GW150914",
    1135136350: "GW151226",
    1167559936: "GW170104",
    1180922494: "GW170608",
    1186741861: "GW170814",
}

In [23]:
def load_file(filepath):
    with h5py.File(filepath, "r") as f:

        strain = f["strain"]["Strain"][:]

        gps_start = int(f["meta"]["GPSstart"][()])

        duration = int(f["meta"]["Duration"][()])

    return strain, gps_start, duration

In [24]:
EVENTS = {
    1126256640: {
        "name": "GW150914",
        "event_gps": 1126259462
    },
    1135132672: {
        "name": "GW151226",
        "event_gps": 1135136350
    },
    1167556608: {
        "name": "GW170104",
        "event_gps": 1167559936
    },
    1180921856: {
        "name": "GW170608",
        "event_gps": 1180922494
    },
    1186738176: {
        "name": "GW170814",
        "event_gps": 1186741861
    }
}

In [25]:
def create_labels(gps_start, duration, window_size=1):
    """
    Creates one label per 1-second window.
    1 = window contains GW event
    0 = noise
    """

    num_windows = duration // window_size

    labels = np.zeros(num_windows, dtype=np.int32)

    if gps_start in EVENTS:

        event_second = EVENTS[gps_start]["event_gps"] - gps_start

        window_index = int(event_second // window_size)

        if 0 <= window_index < num_windows:
            labels[window_index] = 1

    return labels

In [26]:
strain, gps_start, duration = load_file(event_files[0])

labels = create_labels(gps_start, duration)

print("GPS Start:", gps_start)
print("Duration:", duration)

print("Number of labels:", len(labels))

print("Positive labels:", np.sum(labels))

print("Positive window index:", np.where(labels == 1)[0])

GPS Start: 1167556608
Duration: 4096
Number of labels: 4096
Positive labels: 1
Positive window index: [3328]


In [27]:
all_labels = []

for filepath in event_files:

    strain, gps_start, duration = load_file(filepath)

    labels = create_labels(gps_start, duration)

    print(f"{EVENTS[gps_start]['name']}")
    print(f"GPS Start: {gps_start}")
    print(f"Positive Window: {np.where(labels == 1)[0][0]}")
    print("-" * 40)

    all_labels.append(labels)

GW170104
GPS Start: 1167556608
Positive Window: 3328
----------------------------------------
GW170814
GPS Start: 1186738176
Positive Window: 3685
----------------------------------------
GW150914
GPS Start: 1126256640
Positive Window: 2822
----------------------------------------
GW151226
GPS Start: 1135132672
Positive Window: 3678
----------------------------------------
GW170608
GPS Start: 1180921856
Positive Window: 638
----------------------------------------


In [28]:
labels = np.concatenate(all_labels)

print("Labels shape:", labels.shape)

print("Positive samples:", np.sum(labels))

print("Negative samples:", len(labels) - np.sum(labels))

Labels shape: (20480,)
Positive samples: 5
Negative samples: 20475


In [29]:
import os

os.makedirs("../data/datasets", exist_ok=True)

np.save("../data/datasets/event_windows.npy", dataset)
np.save("../data/datasets/labels.npy", labels)

print("Dataset and labels saved successfully!")

Dataset and labels saved successfully!


In [30]:
window_metadata = []

for filepath in event_files:

    _, gps_start, duration = load_file(filepath)

    num_windows = duration

    for window_index in range(num_windows):

        window_metadata.append({
            "gps_start": gps_start,
            "window_index": window_index,
            "event_name": EVENTS[gps_start]["name"]
        })

print("Metadata entries:", len(window_metadata))

Metadata entries: 20480


In [31]:
import pandas as pd

metadata_df = pd.DataFrame(window_metadata)

metadata_df.to_csv(
    "../data/datasets/window_metadata.csv",
    index=False
)

print(metadata_df.head())


    gps_start  window_index event_name
0  1167556608             0   GW170104
1  1167556608             1   GW170104
2  1167556608             2   GW170104
3  1167556608             3   GW170104
4  1167556608             4   GW170104
